# 📊 Notebook 01 — Exploration & Nettoyage
**Projet** : Analyse des déterminants de la satisfaction en maison de retraite
**Jeu de données** : Retraite.xlsx — 300 résidents, 16 variables
**Objectif** : Charger, inspecter et nettoyer les données

## 0. Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys
sys.path.append('..')
from src.preprocessing import load_data, inspect_data, clean_data, detect_outliers, save_data

pd.set_option('display.max_columns', None)
sns.set_theme(style='whitegrid')
print('✅ Imports OK')

## 1. Chargement des données

In [ ]:
df = load_data('../data/raw/Retraite.xlsx')
df.head()

## 2. Structure du jeu de données

In [ ]:
print(f'Dimensions : {df.shape[0]} résidents x {df.shape[1]} variables')
print('\nTypes de colonnes :')
print(df.dtypes)
print('\nStatistiques descriptives :')
df.describe().round(2)

## 3. Valeurs manquantes

In [ ]:
manquants = df.isnull().sum()
manquants = manquants[manquants > 0]

fig, ax = plt.subplots(figsize=(8, 4))
manquants.plot(kind='bar', color='salmon', ax=ax)
ax.set_title('Valeurs manquantes par variable')
ax.set_ylabel('Nombre de valeurs manquantes')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('../reports/figures/valeurs_manquantes.png', dpi=150)
plt.show()
print(manquants)

**Interprétation** : Les variables `Reconfort`, `Confort_Chambre` et `Services_Annexes` présentent des valeurs manquantes.
→ On applique une imputation par la **médiane** (robuste aux valeurs extrêmes).

## 4. Nettoyage — Imputation par la médiane

In [ ]:
df_clean = clean_data(df)
print('\nValeurs manquantes après nettoyage :')
print(df_clean.isnull().sum().sum(), '<-- doit être 0')

## 5. Détection des outliers (méthode IQR)

In [ ]:
rapport_outliers = detect_outliers(df_clean)

## 6. Distribution des variables qualitatives

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
variables_quali = ['Sexe', 'Public', 'Formule', 'CSP']

for ax, var in zip(axes.flatten(), variables_quali):
    counts = df_clean[var].value_counts()
    counts.plot(kind='bar', ax=ax, color='steelblue', edgecolor='white')
    ax.set_title(f'Distribution — {var}')
    ax.set_ylabel('Effectif')
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha='right')

plt.suptitle('Distribution des variables qualitatives', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/figures/distribution_quali.png', dpi=150)
plt.show()

## 7. Distribution des variables numériques (services)

In [ ]:
cols_services = ['Accueil', 'Soins_Qualite', 'Competence_Personnel',
    'Disponibilite_Personnel', 'Reconfort', 'Qualite_Restauration',
    'Hygiene', 'Confort_Chambre', 'Services_Annexes']

fig, axes = plt.subplots(3, 3, figsize=(14, 10))

for ax, col in zip(axes.flatten(), cols_services):
    df_clean[col].plot(kind='hist', bins=5, ax=ax, color='steelblue', edgecolor='white', density=True)
    ax.set_title(col)
    ax.set_xlabel('Note (sur 5)')
    ax.set_ylabel('Densité')

plt.suptitle('Distribution des notes de services (sur 5)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/figures/distribution_services.png', dpi=150)
plt.show()

## 8. Sauvegarde des données nettoyées

In [ ]:
save_data(df_clean, '../data/processed/retraite_clean.csv')
print('✅ Notebook 01 terminé — passer au notebook 02_metriques.ipynb')